In [4]:
from pathlib import Path
import unicodedata
import pandas as pd
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
carpeta = raiz / "data" / "extracted" / "txt" / "osiptel_reportes"
print("Proyecto:", raiz)
print("Carpeta de TXT:", carpeta)
if not carpeta.is_dir():
    raise FileNotFoundError("No se encuentra la carpeta de reportes. Revisa desde dónde abriste JupyterLab.")
archivos = sorted(carpeta.rglob("*.txt"))
print("Reportes TXT encontrados:", len(archivos))
print("Archivos con contenido:", sum(p.stat().st_size > 0 for p in archivos))
if not archivos:
    raise ValueError("No se encontraron archivos TXT.")

Proyecto: /home/nevado/BigData_UPAO/Osiptel_Proyecto
Carpeta de TXT: /home/nevado/BigData_UPAO/Osiptel_Proyecto/data/extracted/txt/osiptel_reportes
Reportes TXT encontrados: 105
Archivos con contenido: 105


In [14]:
def normalizar(texto):
    sin_tildes = unicodedata.normalize("NFD", texto.casefold())
    return "".join(c for c in sin_tildes if unicodedata.category(c) != "Mn")
terminos = ["internet fijo", "fibra optica", "conexiones", "departamento", "penetracion", "cobertura", "despliegue"]
filas = []
for archivo in archivos:
    texto = archivo.read_text(encoding="utf-8", errors="replace")
    t = normalizar(texto)
    conteos = {palabra.replace(" ", "_"): t.count(palabra) for palabra in terminos}
    filas.append({"archivo": archivo.name, "ruta_txt": str(archivo.relative_to(carpeta)), "caracteres": len(texto), **conteos})
resumen = pd.DataFrame(filas)
cols_terminos = [palabra.replace(" ", "_") for palabra in terminos]
resumen["temas_detectados"] = (resumen[cols_terminos] > 0).sum(axis=1)
resumen["menciones"] = resumen[cols_terminos].sum(axis=1)
resumen = resumen.sort_values(["temas_detectados", "menciones", "caracteres"], ascending=False).reset_index(drop=True)
print("Reportes con alguna coincidencia:", int((resumen['menciones'] > 0).sum()))
display(resumen[["archivo", "caracteres", "temas_detectados", "menciones"]].head(15))

Reportes con alguna coincidencia: 67


,archivo,caracteres,temas_detectados,menciones
0,5905973-reporte-estadistico-n-07-agosto-2024.txt,8718,6,49
1,5905919-reporte-estadistico-n-01-febrero-2024.txt,5685,5,42
2,5905857-reporte-estadistico-n-06-agosto-2023.txt,9287,4,50
3,7802224-reporte-estadistico-n-06.txt,4463,4,46
4,5905957-reporte-estadistico-n-05-mayo-2024.txt,9639,4,42
5,6644026-reporte-estadistico-n-11-noviembre-202...,8538,4,41
6,7802468-reporte-estadistico-n-01.txt,5385,4,35
7,5905738-reporte-estadistico-agosto-2022.txt,7070,4,33
8,5905760-reporte-estadistico-n-01-febrero-2023.txt,7498,4,29
9,5905845-reporte-estadistico-n-05-mayo-2023.txt,8402,4,23


In [15]:
palabras_busqueda = ["fibra optica", "internet fijo", "conexiones"]
seleccionados = resumen.loc[resumen["menciones"] > 0, "ruta_txt"].head(3).tolist()
for ruta in seleccionados:
    archivo = carpeta / ruta
    texto = archivo.read_text(encoding="utf-8", errors="replace")
    t = normalizar(texto)
    print("\n" + "=" * 72)
    print("DOCUMENTO:", archivo.name)
    mostrados = 0
    for palabra in palabras_busqueda:
        posicion = t.find(palabra)
        if posicion >= 0:
            inicio, fin = max(0, posicion - 130), min(len(texto), posicion + 250)
            print(f"\nPalabra: {palabra} | Fragmento:")
            print(" ".join(texto[inicio:fin].split()))
            mostrados += 1
        if mostrados == 2:
            break
    if not mostrados:
        print("No se hallaron fragmentos para las palabras priorizadas.")


DOCUMENTO: 5905973-reporte-estadistico-n-07-agosto-2024.txt

Palabra: fibra optica | Fragmento:
AGOSTO 2024 / Nº 07 Conexiones de internet fijo con fibra óptica crecieron 61.66 % al primer semestre de 2024 Al cierre de junio de 2024, las empresas operadoras registraron 2.5 millones de conexiones con fibra óptica y representan el 65.45 % del total de conexiones de in

Palabra: internet fijo | Fragmento:
AGOSTO 2024 / Nº 07 Conexiones de internet fijo con fibra óptica crecieron 61.66 % al primer semestre de 2024 Al cierre de junio de 2024, las empresas operadoras registraron 2.5 millones de conexiones con fibra óptica y representan el 65.45 % del total d

DOCUMENTO: 5905919-reporte-estadistico-n-01-febrero-2024.txt

Palabra: fibra optica | Fragmento:
FEBRERO 2024 / Nº 01 OSIPTEL: internet fijo con fibra óptica superó los 2 millones de conexiones a nivel nacional en 2023 A diciembre de 2023, las conexiones de fibra óptica se consolidaron como el medio más utilizado para la provisión del 

In [16]:
import pandas as pd
resultados = []

for archivo in archivos:
    texto = normalizar(
        archivo.read_text(
            encoding="utf-8",
            errors="replace"
        )
    )
    tiene_internet_fijo = "internet fijo" in texto
    tiene_fibra = "fibra optica" in texto
    tiene_conexiones = "conexiones" in texto
    if tiene_internet_fijo and tiene_fibra:
        categoria = "Prioridad alta"
    elif tiene_internet_fijo:
        categoria = "Prioridad media"
    else:
        categoria = "Revisar relevancia"
    resultados.append({
        "documento": archivo.name,
        "internet_fijo": tiene_internet_fijo,
        "fibra_optica": tiene_fibra,
        "conexiones": tiene_conexiones,
        "categoria": categoria
    })
df_clasificacion = pd.DataFrame(resultados)
print("TOTAL DOCUMENTOS:", len(df_clasificacion))
print("\nCLASIFICACIÓN:")
print(df_clasificacion["categoria"].value_counts())
print("\nDOCUMENTOS DE PRIORIDAD ALTA:")
display(
    df_clasificacion[
        df_clasificacion["categoria"] == "Prioridad alta"
    ][["documento", "categoria"]]
)

TOTAL DOCUMENTOS: 105

CLASIFICACIÓN:
categoria
Revisar relevancia    67
Prioridad alta        21
Prioridad media       17
Name: count, dtype: int64

DOCUMENTOS DE PRIORIDAD ALTA:


,documento,categoria
15,5905738-reporte-estadistico-agosto-2022.txt,Prioridad alta
17,5905760-reporte-estadistico-n-01-febrero-2023.txt,Prioridad alta
20,5905785-reporte-estadistico-n-04-mayo-2023.txt,Prioridad alta
21,5905845-reporte-estadistico-n-05-mayo-2023.txt,Prioridad alta
22,5905857-reporte-estadistico-n-06-agosto-2023.txt,Prioridad alta
26,5905897-reporte-estadistico-n-10-noviembre-202...,Prioridad alta
28,5905919-reporte-estadistico-n-01-febrero-2024.txt,Prioridad alta
31,5905944-reporte-estadistico-n-04-mayo-2024.txt,Prioridad alta
32,5905957-reporte-estadistico-n-05-mayo-2024.txt,Prioridad alta
34,5905973-reporte-estadistico-n-07-agosto-2024.txt,Prioridad alta


In [17]:
import pandas as pd
ruta_excel = raiz / "data/raw/excel/2.1 CONEXIONES DE INTERNET FIJO.xlsx"
df_osiptel = pd.read_excel(
    ruta_excel,
    sheet_name="Dataset",
    header=3,
    usecols="B:H"
)
df_osiptel["Mes"] = pd.to_datetime(
    df_osiptel["Mes"], errors="coerce"
)
df_osiptel["Conexiones"] = pd.to_numeric(
    df_osiptel["Conexiones"], errors="coerce"
)
print("COMPARACIÓN CON REPORTES OFICIALES DE OSIPTEL")
for periodo in ["2023-12", "2024-06"]:
    datos = df_osiptel[
        df_osiptel["Mes"].dt.strftime("%Y-%m") == periodo
    ]
    fibra = datos[
        datos["Tecnologia"].astype(str).str.contains(
            "Fibra", case=False, na=False
        )
    ]
    total_conexiones = datos["Conexiones"].sum()
    conexiones_fibra = fibra["Conexiones"].sum()
    porcentaje = (
        conexiones_fibra / total_conexiones * 100
        if total_conexiones > 0 else 0
    )
    print("\n" + "=" * 55)
    print("PERIODO:", periodo)
    print("Filas encontradas:", len(datos))
    print("Total internet fijo:", f"{total_conexiones:,.0f}")
    print("Conexiones fibra óptica:", f"{conexiones_fibra:,.0f}")
    print("Participación fibra óptica:", f"{porcentaje:.2f}%")

COMPARACIÓN CON REPORTES OFICIALES DE OSIPTEL

PERIODO: 2023-12
Filas encontradas: 4814
Total internet fijo: 3,563,247
Conexiones fibra óptica: 2,038,468
Participación fibra óptica: 57.21%

PERIODO: 2024-06
Filas encontradas: 4729
Total internet fijo: 3,800,822
Conexiones fibra óptica: 2,524,774
Participación fibra óptica: 66.43%


In [18]:
import re
import pandas as pd
ruta_reporte = carpeta / "5905973-reporte-estadistico-n-07-agosto-2024.txt"
texto = " ".join(
    ruta_reporte.read_text(
        encoding="utf-8",
        errors="replace"
    ).split()
)
print("FRAGMENTO DEL REPORTE OFICIAL\n")
coincidencia = re.search(r"65[.,]45", texto)
if coincidencia:
    inicio = max(0, coincidencia.start() - 400)
    fin = min(len(texto), coincidencia.end() + 400)
    print(texto[inicio:fin])
else:
    print("No se encontró el porcentaje 65.45 en el TXT.")
datos = df_osiptel[
    df_osiptel["Mes"].dt.strftime("%Y-%m") == "2024-06"
].copy()

es_fibra = datos["Tecnologia"].astype(str).str.contains(
    "fibra", case=False, na=False
)
datos["conexiones_fibra"] = datos["Conexiones"].where(
    es_fibra, 0
)
resumen_segmento = (
    datos.groupby("Segmento", dropna=False)
    .agg(
        total_conexiones=("Conexiones", "sum"),
        conexiones_fibra=("conexiones_fibra", "sum")
    )
    .reset_index()
)
resumen_segmento["porcentaje_fibra"] = (
    resumen_segmento["conexiones_fibra"]
    / resumen_segmento["total_conexiones"]
    * 100
).round(2)
print("\nCONEXIONES POR SEGMENTO - JUNIO 2024")
display(resumen_segmento)

FRAGMENTO DEL REPORTE OFICIAL

AGOSTO 2024 / Nº 07 Conexiones de internet fijo con fibra óptica crecieron 61.66 % al primer semestre de 2024 Al cierre de junio de 2024, las empresas operadoras registraron 2.5 millones de conexiones con fibra óptica y representan el 65.45 % del total de conexiones de internet fijo a nivel nacional. Por empresas operadoras, el Grupo Win y Wow consolidan su presencia y en conjunto suman más del 41 % de participación en el mercado de internet fijo con fibra óptica. La penetración de la fibra óptica en el mercado de internet fijo sigue creciendo sostenidamente, impulsada por la demanda de mayores velocidades de acceso en este servicio

CONEXIONES POR SEGMENTO - JUNIO 2024


,Segmento,total_conexiones,conexiones_fibra,porcentaje_fibra
0,Comercial,189938,85326,44.92
1,Residencial,3610884,2439448,67.56


In [19]:
datos_2026 = df_osiptel[
    df_osiptel["Mes"].dt.strftime("%Y-%m") == "2026-03"
].copy()

fibra_2026 = datos_2026[
    datos_2026["Tecnologia"].astype(str).str.contains(
        "fibra", case=False, na=False
    )
]
total = datos_2026["Conexiones"].sum()
total_fibra = fibra_2026["Conexiones"].sum()
print("INTERNET FIJO - MARZO 2026")
print("Total de conexiones:", f"{total:,.0f}")
print("Conexiones de fibra óptica:", f"{total_fibra:,.0f}")
print("Participación de fibra:", f"{total_fibra / total * 100:.2f}%")
print("\nDEPARTAMENTOS CON MÁS CONEXIONES DE FIBRA")
resultado = (
    fibra_2026.groupby("Departamento")["Conexiones"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
display(resultado)

INTERNET FIJO - MARZO 2026
Total de conexiones: 4,135,995
Conexiones de fibra óptica: 3,414,062
Participación de fibra: 82.55%

DEPARTAMENTOS CON MÁS CONEXIONES DE FIBRA


Departamento
LIMA           1770004
LA LIBERTAD     177270
AREQUIPA        166708
LAMBAYEQUE      154685
PIURA           150747
CALLAO          137490
ICA             133799
ANCASH           97947
JUNIN            87115
CUSCO            77996
Name: Conexiones, dtype: int64

In [20]:
import re
import pandas as pd
resultados_recientes = []
for archivo in archivos:
    texto = normalizar(
        " ".join(
            archivo.read_text(
                encoding="utf-8",
                errors="replace"
            ).split()
        )
    )
    if "internet fijo" not in texto:
        continue
    if "fibra optica" not in texto:
        continue
    for coincidencia in re.finditer("fibra optica", texto):
        inicio = max(0, coincidencia.start() - 250)
        fin = min(len(texto), coincidencia.end() + 350)
        fragmento = texto[inicio:fin]
        if (
            "internet fijo" in fragmento
            and ("2025" in fragmento or "2026" in fragmento)
        ):
            resultados_recientes.append({
                "documento": archivo.name,
                "fragmento": fragmento
            })
            break
df_recientes = pd.DataFrame(resultados_recientes)
print("DOCUMENTOS CON REFERENCIAS A 2025 O 2026:",
      len(df_recientes))
if not df_recientes.empty:
    for _, fila in df_recientes.head(10).iterrows():
        print("\n" + "=" * 65)
        print("DOCUMENTO:", fila["documento"])
        print("FRAGMENTO:", fila["fragmento"])
else:
    print("No se encontraron fragmentos con esos criterios.")

DOCUMENTOS CON REFERENCIAS A 2025 O 2026: 5

DOCUMENTO: 7801986-reporte-estadistico-n-04-mayo-2025.txt
FRAGMENTO: rca del 12 % al cierre del primer trimestre de 2025 el numero de conexiones de internet fijo alcanzo los 4.1 millones al cierre de marzo de 2025. wow fue tercero en el mercado y desplazo a win. mas del 77 % del total conexiones de internet fijo usan fibra optica. alrededor del 91 % de conexiones contrato velocidades de descarga mayores o iguales a 100 mbps. el servicio de acceso a internet fijo sigue avanzando y al cierre del primer trimestre de 2025, peru reporto un nuevo pico historico al bordear los 4.1 millones de conexiones de internet fijo, 11.9 % superior a la suma reportada en el mismo periodo del

DOCUMENTO: 7802224-reporte-estadistico-n-06.txt
FRAGMENTO: reporte estadistico setiembre 2025 | n.o 6 alrededor del 80 % de conexiones de internet fijo usan fibra optica accesos a internet fijo superaron los 4.19 millones de conexiones al cierre de junio de 2025, gracias 

In [21]:
import pandas as pd
referencias = [
    {
        "periodo": "2025-09",
        "reporte": "Noviembre 2025",
        "conexiones_publicadas": 4274935,
        "fibra_publicada_pct": None
    },
    {
        "periodo": "2025-12",
        "reporte": "Febrero 2026",
        "conexiones_publicadas": None,
        "fibra_publicada_pct": 82.47
    },
    {
        "periodo": "2026-03",
        "reporte": "Primer trimestre 2026",
        "conexiones_publicadas": None,
        "fibra_publicada_pct": 82.55
    }
]
resultados = []
for ref in referencias:
    datos = df_osiptel[
        df_osiptel["Mes"].dt.strftime("%Y-%m")
        == ref["periodo"]
    ]
    total = datos["Conexiones"].sum()
    fibra = datos[
        datos["Tecnologia"].astype(str).str.contains(
            "fibra", case=False, na=False
        )
    ]["Conexiones"].sum()
    porcentaje = round(fibra / total * 100, 2) if total else None
    resultados.append({
        "Periodo": ref["periodo"],
        "Reporte": ref["reporte"],
        "Conexiones Excel": int(total),
        "Conexiones PDF exactas": ref["conexiones_publicadas"],
        "Fibra Excel (%)": porcentaje,
        "Fibra PDF (%)": ref["fibra_publicada_pct"]
    })
comparacion = pd.DataFrame(resultados)
display(comparacion)

,Periodo,Reporte,Conexiones Excel,Conexiones PDF exactas,Fibra Excel (%),Fibra PDF (%)
0,2025-09,Noviembre 2025,4289604,4274935.0,81.13,NaN
1,2025-12,Febrero 2026,4379544,NaN,82.53,82.47
2,2026-03,Primer trimestre 2026,4135995,NaN,82.55,82.55


In [22]:
import pandas as pd
datos = df_osiptel.copy()
datos["periodo"] = datos["Mes"].dt.strftime("%Y-%m")
datos = datos[
    datos["periodo"].isin(["2025-12", "2026-03"])
]
por_tecnologia = datos.pivot_table(
    index="Tecnologia",
    columns="periodo",
    values="Conexiones",
    aggfunc="sum",
    fill_value=0
)
por_tecnologia["diferencia"] = (
    por_tecnologia["2026-03"]
    - por_tecnologia["2025-12"]
)
print("CAMBIO POR TECNOLOGÍA")
display(por_tecnologia.sort_values("diferencia"))
por_empresa = datos.pivot_table(
    index="Empresa",
    columns="periodo",
    values="Conexiones",
    aggfunc="sum",
    fill_value=0
)
por_empresa["diferencia"] = (
    por_empresa["2026-03"]
    - por_empresa["2025-12"]
)
print("\nEMPRESAS CON MAYORES REDUCCIONES")
display(
    por_empresa.sort_values("diferencia").head(10)
)

CAMBIO POR TECNOLOGÍA


periodo,2025-12,2026-03,diferencia
Tecnologia,,,
5) Fibra Óptica,3614437,3414062,-200375
3) Cablemodem,628544,583641,-44903
7) Otras Tecnologías,54342,50481,-3861
2) xDSL,12965,11341,-1624
4) Wimax,156,144,-12
6) Satelital,69100,76326,7226



EMPRESAS CON MAYORES REDUCCIONES


periodo,2025-12,2026-03,diferencia
Empresa,,,
WOW TEL S.A.C.,809557,505271,-304286
INTEGRATEL PERU S.A.A.,1064839,1030003,-34836
VIETTEL PERU S.A.C.,64171,51591,-12580
CABLENORTV S.A.C.,2300,434,-1866
ENTEL PERU S.A.,29875,28323,-1552
ZAAZ PERU S.A.C.,67997,66844,-1153
DIRECTV PERU S.R.L.,11250,10563,-687
HUGHES DE PERU S.R.L.,5224,4580,-644
SORITV S.A.C.,2743,2468,-275
